# Training MetricGAN+KAN
Mainly based on train.py from MetricGAN+.

In [ ]:
!pip install speechbrain
!pip install hyperpyyaml
!pip install pesq
!pip install pysepm
!pip install git+https://github.com/schmiph2/pysepm.git
!pip install tensorboard

In [ ]:
import os
import copy
import shutil
import sys
from enum import Enum, auto

import torch
import torchaudio
from hyperpyyaml import load_hyperpyyaml
from pesq import pesq

import speechbrain as sb
from speechbrain.dataio.sampler import ReproducibleWeightedRandomSampler
from speechbrain.nnet.loss.stoi_loss import stoi_loss
from speechbrain.processing.features import spectral_magnitude
from speechbrain.utils.distributed import run_on_main
from speechbrain.utils.metric_stats import MetricStats

# from MetricGAN_KAN import MetricDiscriminator, EnhancementGenerator

from pysepm import composite
from train import *
from spectrogram import *

N_fft = 512
data_folder = "../data/noisy-vctk-16k"
output_folder = "../results"
figures_folder = "../figures"

In [ ]:
class PerceptualDistillation:
    def __init__(self, generator, sample_rate=16000):
        self.generator_old = copy.deepcopy(generator)
        self.generator_old.eval()
        for p in self.generator_old.parameters():
            p.requires_grad_(False)
        self.sample_rate = sample_rate

    def distillation_loss(self, noisy_wavs, new_generator):
        device = noisy_wavs.device
        hop = N_fft // 4
        win = torch.hann_window(N_fft).to(device)

        def to_mag(wavs):
            return torch.stft(
                wavs, n_fft=N_fft, hop_length=hop, win_length=N_fft,
                window=win, return_complex=True,
            ).abs()

        def to_wav(mask, noisy_mag, noisy_wav):
            noisy_stft = torch.stft(
                noisy_wav, n_fft=N_fft, hop_length=hop, win_length=N_fft,
                window=win, return_complex=True,
            )
            phase = noisy_stft / (noisy_stft.abs() + 1e-8)
            return torch.istft(
                mask * noisy_mag * phase,
                n_fft=N_fft, hop_length=hop, win_length=N_fft, window=win,
            )

        noisy_mag = to_mag(noisy_wavs)
        noisy_mag_t = noisy_mag.permute(0, 2, 1)

        with torch.no_grad():
            wav_old = to_wav(
                self.generator_old(noisy_mag_t).permute(0, 2, 1),
                noisy_mag, noisy_wavs,
            ).detach()

        wav_new = to_wav(
            new_generator(noisy_mag_t).permute(0, 2, 1),
            noisy_mag, noisy_wavs,
        )

        return stoi_loss(wav_new, wav_old, self.sample_rate)

In [ ]:
class MGKBrainPD(MGKBrain):
    def __init__(self, *args, perceptual_distillation=None, pd_lambda=400.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.perceptual_distillation = perceptual_distillation
        self.pd_lambda = pd_lambda

    def compute_objectives(self, predictions, batch, stage, optim_name=""):
        loss = super().compute_objectives(predictions, batch, stage, optim_name=optim_name)
        if (
            stage == sb.Stage.TRAIN
            and self.sub_stage == SubStage.GENERATOR
            and self.perceptual_distillation is not None
        ):
            noisy_wavs, _ = batch.noisy_sig
            pd_loss = self.perceptual_distillation.distillation_loss(
                noisy_wavs, self.modules.generator
            )
            self.hparams.train_logger.log_stats({"PD loss (unscaled)": pd_loss.item()})
            loss = loss + self.pd_lambda * pd_loss
        return loss

# Requirements
```bash
pip install speechbrain
pip install https://github.com/schmiph2/pysepm/archive/master.zip
```

If `kaiser` is not found leading to an `ImportError`, you may need to go to line 2 of /path/to/your/python_env/site-packages/pysepm/utils.py, and change
```python
from scipy.signal import firls,kaiser,upfirdn
```
to
```python
from scipy.signal import firls,upfirdn
from scipy.signal.windows import kaiser
```

# Training
Mainly based on train.py from MetricGAN+.

In [ ]:
def train_and_eval(hparams_file, perceptual_distillation=None, pd_lambda=400.0):
    # hparams_file, run_opts, _ = sb.parse_arguments(model_param_file)
    run_opts = None
    model_name = hparams_file.split(".")[-2]
    overrides = {"output_folder": os.path.join(output_folder, model_name), "data_folder": data_folder, "number_of_epochs": 1}
    with open(hparams_file) as fin:
        hparams = load_hyperpyyaml(fin, overrides=overrides)

    # Initialize ddp (useful only for multi-GPU DDP training)
    sb.utils.distributed.ddp_init_group(run_opts)

    # Data preparation
    from voicebank_prepare import prepare_voicebank  # noqa

    run_on_main(
        prepare_voicebank,
        kwargs={
            "data_folder": hparams["data_folder"],
            "save_folder": hparams["data_folder"],
            "skip_prep": hparams["skip_prep"],
        },
    )

    # Create dataset objects
    datasets = dataio_prep(hparams)

    # Create experiment directory
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file,
        overrides=overrides,
    )

    if hparams["use_tensorboard"]:
        from speechbrain.utils.train_logger import TensorboardLogger

        hparams["tensorboard_train_logger"] = TensorboardLogger(
            hparams["tensorboard_logs"]
        )

    # Create the folder to save enhanced files (+ support for DDP)
    run_on_main(create_folder, kwargs={"folder": hparams["enhanced_folder"]})

    se_brain = MGKBrainPD(
        modules=hparams["modules"],
        hparams=hparams,
        run_opts=run_opts,
        checkpointer=hparams["checkpointer"],
        perceptual_distillation=perceptual_distillation,
        pd_lambda=pd_lambda,
    )
    se_brain.train_set = datasets["train"]
    se_brain.batch_size = hparams["dataloader_options"]["batch_size"]
    se_brain.sub_stage = SubStage.GENERATOR

    # Load latest checkpoint to resume training
    se_brain.fit(
        epoch_counter=se_brain.hparams.epoch_counter,
        train_set=datasets["train"],
        valid_set=datasets["valid"],
        train_loader_kwargs=hparams["dataloader_options"],
        valid_loader_kwargs=hparams["valid_dataloader_options"],
    )

    # Load best checkpoint for evaluation
    test_stats = se_brain.evaluate(
        test_set=datasets["test"],
        max_key=hparams["target_metric"],
        test_loader_kwargs=hparams["dataloader_options"],
    )

    # Show parameters summary
    param_count = sum(p.numel() for p in se_brain.modules.generator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Generator parameter count": param_count})
    param_count = sum(p.numel() for p in se_brain.modules.discriminator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Discriminator parameter count": param_count})

    return se_brain, datasets

Loop through the model files and train them.

In [ ]:
PD_LAMBDA = 400.0  # 100 = more plasticity, 1000 = more retention

"""task_yamls = [
    f"hparams/train_{fname}"
    for fname in sorted(os.listdir("models"))
    if fname.endswith(".yaml")
]
"""
task_yamls = ["hparams/train_mgk_g4.yaml"]

pd = None

for idx, yaml_file in enumerate(task_yamls):
    print(f"Task {idx + 1}/{len(task_yamls)}: {yaml_file}  |  PD active: {pd is not None}")
    brain, datasets = train_and_eval(yaml_file, perceptual_distillation=pd, pd_lambda=PD_LAMBDA)
    pd = PerceptualDistillation(brain.modules.generator)

Generate spectrograms.

In [ ]:
model_name_list = [fname.split(".")[-2] for fname in os.listdir("models") if fname.endswith(".yaml")]
enh_folder = [f"{output_folder}/{fname.split(".")[-2]}/enhanced_wavs" for fname in os.listdir("models") if fname.endswith(".yaml")]
# os.path.join(output_folder, model_name)
for m in model_name_list:
    target_dir = f"{figures_folder}/{m}"
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    save_spec(f"{output_folder}/{m}/enhanced_wavs", target_dir)

Codes for generating model's hyper-param files.

In [ ]:
def generate_train_config():
    templates = None
    with open("hparams/train.yaml") as f:
        templates = f.readlines()
    for fname in os.listdir("models"):
        tmp = fname.split(".")
        if tmp[-1] == "yaml":
            with open(f"hparams/train_{tmp[-2]}.yaml", "w") as f:
                spec = templates.copy()
                spec[64] = f"models: !include:../models/{fname}\n"
                f.write("# Auto generated basing on train.yaml\n")
                f.writelines(spec)
            
# generate_train_config()